## 🚀 Colab Pro Setup — chạy **CELL NÀY TRƯỚC** mỗi lần mở notebook

Giả định:
- Đã chạy `colab_setup_frl.ipynb` ít nhất **một lần** để cài venv vào `MyDrive/BigData/venv_frl`
- Notebook này nằm đâu đó trong Drive (khuyến nghị `MyDrive/BigData/notebooks/`)
- Runtime: **GPU (T4 trở lên)**

Cell dưới sẽ: mount Drive → thêm venv vào `sys.path` → set paths vào Drive + local cache.


In [1]:
# ================================================================
# COLAB SETUP: cài thư viện + Mount Drive + Set paths
# ================================================================
print("⏳ Đang cài đặt thư viện vào hệ thống Colab, vui lòng chờ...")

!pip install --no-cache-dir \
    "numpy<2.0.0" \
    "pandas>=2.1.0" \
    "torch>=2.2.0" \
    "citylearn==2.5.0" \
    "stable-baselines3>=2.2.1" \
    "gymnasium==0.28.1" \
    "tensorboard"

print("\n✅ Đã cài xong thư viện.")



⏳ Đang cài đặt thư viện vào hệ thống Colab, vui lòng chờ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 100.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of stable-baselines3 to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.5/398.5 kB 321.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 254.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 290.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 426.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3


✅ Đã cài xong thư viện.


In [6]:
# ----------------------------------------------------------------
# Mount Drive + set DRIVE_ROOT cho run này
# ----------------------------------------------------------------
import os
from pathlib import Path
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("✓ Drive đã mount sẵn")

# ⭐ Thư mục Drive RIÊNG BIỆT cho run này (không ghi đè run cũ)
DRIVE_ROOT  = Path('/content/drive/MyDrive/BigData/xxxfedrl_nowarm')
LOCAL_CACHE = Path('/content/_frl_nowarm_cache')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

print(f"📁 Drive folder : {DRIVE_ROOT}")
print(f"📁 Local cache  : {LOCAL_CACHE}")

# Verify GPU
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ KHÔNG có GPU")


✓ Drive đã mount sẵn
📁 Drive folder : /content/drive/MyDrive/BigData/xxxfedrl_nowarm
📁 Local cache  : /content/_frl_nowarm_cache
✅ GPU: Tesla T4


# Federated Reinforcement Learning (FedRL) — SAC cho Smart Grid

**Giai đoạn 2** trong lộ trình: `Centralized SAC → MARL IL → FedRL ← YOU ARE HERE`

## Tổng quan kiến trúc

```
╔══════════════════════════════════════════════════════╗
║              FEDERATED SERVER (Aggregator)           ║
║    w_global = FedAvg(w₁, w₂, w₃, w₄, w₅, w₆)       ║
╚═══════╦══════════════════════════════════╦═══════════╝
        ║ broadcast w_global               ║ receive w_i
        ▼                                  ▲
  ┌─────────┐  ┌─────────┐  ...  ┌─────────┐
  │  Node 1 │  │  Node 2 │       │  Node 6 │
  │ SAC_B1  │  │ SAC_B2  │       │ SAC_B6  │
  │ Env_B1  │  │ Env_B2  │       │ Env_B6  │
  │ data_B1 │  │ data_B2 │       │ data_B6 │
  └─────────┘  └─────────┘       └─────────┘
       ↕              ↕                  ↕
  Local train    Local train        Local train
  (K episodes)   (K episodes)       (K episodes)
```

## Nền tảng lý thuyết

### 1. Mục tiêu tối ưu hóa toàn cục

FedRL giải quyết bài toán:

$$\min_w F(w) = \sum_{i=1}^{N} \frac{n_i}{n} F_i(w)$$

Trong đó $F_i(w)$ là **objective function cục bộ** của agent $i$:

$$F_i(w) = -\mathbb{E}_{\tau \sim \pi_w^i} \left[ \sum_t r^i_t + \alpha \mathcal{H}(\pi_w^i(\cdot|o^i_t)) \right]$$

### 2. FedAvg Algorithm

Thuật toán **Federated Averaging (McMahan et al., 2017)** cho RL:

$$w^{r+1}_{global} = \sum_{i=1}^{N} \frac{n_i}{n} \cdot w^r_i$$

Sau đó broadcast: $w^{r+1}_i \leftarrow w^{r+1}_{global}$ (hoặc fine-tune thêm từ $w_{global}$)

| Ký hiệu | Ý nghĩa |
|---|---|
| $r$ | Round federation (1, 2, ..., R) |
| $w^r_i$ | Weights của agent $i$ sau round $r$ |
| $n_i$ | Số local steps agent $i$ thực hiện |
| $n = \sum n_i$ | Tổng local steps |
| $K$ | Số local episodes mỗi round |

### 3. Tại sao FedAvg giải quyết Non-stationarity?

Trong MARL IL, mỗi agent học trong môi trường "trôi nổi" vì các agent khác thay đổi. FedAvg đồng bộ hóa định kỳ:

$$\underbrace{w_i^r}_{\text{local policy}} \xrightarrow{\text{FedAvg}} \underbrace{w_{global}^r}_{\text{consensus}} \xrightarrow{\text{broadcast}} \underbrace{w_i^{r+1}}_{\text{new local start}}$$

Điều này tạo ra **implicit coordination** — mỗi agent "biết" về các agent khác qua shared weights, không cần share data thực tế.

### 4. Privacy Guarantee

Mỗi node chỉ chia sẻ **model weights** $w_i$, không chia sẻ:
- Raw data (điện tiêu thụ, nhiệt độ)
- Observations $o^i_t$
- Rewards $r^i_t$

Với **Differential Privacy** (phần mở rộng), thêm Gaussian noise trước khi upload:
$$\tilde{w}_i = w_i + \mathcal{N}(0, \sigma^2 \mathbf{I}), \quad \sigma = \frac{2 \cdot S \cdot \sqrt{2 \ln(1.25/\delta)}}{\epsilon}$$

### 5. FedProx (Cải tiến FedAvg)

**FedProx** (Li et al., 2020) thêm proximal term để ổn định khi data heterogeneous:

$$F_i^{prox}(w) = F_i(w) + \frac{\mu}{2} \|w - w_{global}\|^2$$

Giữ local weights gần với global → giảm client drift.

In [7]:
import os
import glob
import copy
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# SB3
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.monitor import Monitor

# CityLearn
from citylearn.citylearn import CityLearnEnv
from citylearn.wrappers import NormalizedObservationWrapper, StableBaselines3Wrapper

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} (device: {'cuda' if torch.cuda.is_available() else 'cpu'})")


PyTorch  : 2.10.0+cu128
CUDA     : True (device: cuda)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
# ================================================================
# FEDERATED RL — HYPERPARAMETERS (FAST mode với warm-start MARL)
# ================================================================

# --- Federated settings ---
NUM_HOUSES              = 6       # Số node (tòa nhà)
NUM_ROUNDS              = 30      # ⬇️ 50→25: warm-start nên cần ít rounds hơn
LOCAL_EPISODES_PER_ROUND= 16      # ⬇️ 10→8: warm-start = đỡ cần nhiều local training
EPISODE_LENGTH          = 168     # 1 tuần (giờ)

# --- FedAvg settings ---
AGGREGATION_WEIGHTS     = None    # None = uniform (1/N)

# --- Local SAC settings ---
NET_ARCH                = [256, 256]
BATCH_SIZE              = 512     # ⬆️ tận dụng GPU VRAM
BUFFER_SIZE             = 300_000  # ⬆️ nhớ nhiều hơn
LEARNING_STARTS         = 500
TRAIN_FREQ              = 8       # ⬆️ gom env steps để batch update
GRADIENT_STEPS          = 16      # ⬆️ GPU bận liên tục

# --- VecEnv settings ---
NUM_CPU                 = 4       # Colab CPU bottleneck
USE_SUBPROC             = False   # DummyVecEnv nhanh hơn

# --- Warm-start từ MARL solar_comfort (đã train xong) ---
WARM_START_FROM_MARL    = False    # ⭐ Bật warm-start: buildings có sẵn weights

# --- Checkpoint I/O optimization ---
CHECKPOINT_EVERY        = 5       # backup thường xuyên
RSYNC_EVERY             = 10      # sync Drive đều đặn

# --- Paths ---
BASE_SCHEMA             = "citylearn_challenge_2023_phase_3_1"
SESSION_NAME            = "fedrl_sac_solar_comfort_FROMSCRATCH"

# Fallback nếu chạy trực tiếp
try:
    DRIVE_ROOT, LOCAL_CACHE
except NameError:
    DRIVE_ROOT  = Path.cwd()
    LOCAL_CACHE = Path.cwd() / '_frl_cache'
    LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

# Thư mục đích trên Drive
MODEL_DIR               = DRIVE_ROOT / "models" / SESSION_NAME
TENSORBOARD_DIR         = DRIVE_ROOT / "training_logs"

# ⭐ MARL_MODEL_DIR: trỏ tới MARL solar_comfort (cùng reward function)
MARL_MODEL_DIR          = Path('/content/drive/MyDrive/BigData/xxxmarl_solar_comfort/models/marl_sac_solar_comfort')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
TENSORBOARD_DIR.mkdir(parents=True, exist_ok=True)

# Thư mục staging trên /content (nhanh)
LOCAL_MODEL_DIR         = LOCAL_CACHE / "models" / SESSION_NAME
LOCAL_TB_DIR            = LOCAL_CACHE / "training_logs"
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_TB_DIR.mkdir(parents=True, exist_ok=True)

# Verify MARL warm-start path
if WARM_START_FROM_MARL:
    print(f"\n🔍 Verifying MARL warm-start path...")
    print(f"   {MARL_MODEL_DIR}")
    if MARL_MODEL_DIR.exists():
        final_zips = list(MARL_MODEL_DIR.glob("Building_*/sac_Building_*_final.zip"))
        print(f"   ✓ Tìm thấy {len(final_zips)}/6 MARL final models")
        for fz in sorted(final_zips):
            print(f"     - {fz.parent.name}/{fz.name}")
        if len(final_zips) < 6:
            print(f"   ⚠ Chưa đủ 6 buildings - 1 số agents sẽ train from scratch")
    else:
        print(f"   ❌ Path không tồn tại! Sẽ train from scratch")
        WARM_START_FROM_MARL = False

# Resume FedRL checkpoint nếu có
if any(MODEL_DIR.iterdir()):
    print("\nĐang rsync checkpoints Drive → local cache (để resume)...")
    subprocess.run(
        ["rsync", "-a", f"{MODEL_DIR}/", f"{LOCAL_MODEL_DIR}/"],
        check=False,
    )

N_ENVS                  = NUM_CPU
LOCAL_STEPS_PER_ROUND   = EPISODE_LENGTH * LOCAL_EPISODES_PER_ROUND * N_ENVS
TOTAL_STEPS             = NUM_ROUNDS * LOCAL_STEPS_PER_ROUND * NUM_HOUSES

# ETA estimate
fps_estimate = 125
eta_hours = TOTAL_STEPS / fps_estimate / 3600

print(f"\n{'='*62}")
print(f"  Federated RL — FAST mode (warm-start MARL)")
print(f"{'='*62}")
print(f"  Rounds              : {NUM_ROUNDS}")
print(f"  Agents              : {NUM_HOUSES}")
print(f"  Local episodes/round: {LOCAL_EPISODES_PER_ROUND}")
print(f"  Local steps/round   : {LOCAL_STEPS_PER_ROUND:,}")
print(f"  Total steps         : {TOTAL_STEPS:,}")
print(f"  VecEnv              : {'Subproc' if USE_SUBPROC else 'Dummy'}  (n_envs={N_ENVS})")
print(f"  Batch / Train freq  : {BATCH_SIZE} / {TRAIN_FREQ}")
print(f"  Gradient steps      : {GRADIENT_STEPS}")
print(f"  Warm-start MARL     : {WARM_START_FROM_MARL}")
print(f"  SESSION_NAME        : {SESSION_NAME}")
print(f"  MODEL_DIR (Drive)   : {MODEL_DIR}")
print(f"{'='*62}")
print(f"  📊 ETA dự kiến: ~{eta_hours:.1f}h ({TOTAL_STEPS:,} steps @ {fps_estimate} fps)")
print(f"  💡 Theo dõi: !nvidia-smi để xem GPU usage")
print(f"{'='*62}")



  Federated RL — FAST mode (warm-start MARL)
  Rounds              : 30
  Agents              : 6
  Local episodes/round: 16
  Local steps/round   : 10,752
  Total steps         : 1,935,360
  VecEnv              : Dummy  (n_envs=4)
  Batch / Train freq  : 512 / 8
  Gradient steps      : 16
  Warm-start MARL     : False
  SESSION_NAME        : fedrl_sac_solar_comfort_FROMSCRATCH
  MODEL_DIR (Drive)   : /content/drive/MyDrive/BigData/xxxfedrl_nowarm/models/fedrl_sac_solar_comfort_FROMSCRATCH
  📊 ETA dự kiến: ~4.3h (1,935,360 steps @ 125 fps)
  💡 Theo dõi: !nvidia-smi để xem GPU usage


## FedAvg Aggregator

Class `FedAvgAggregator` triển khai server-side logic:

### Weight Aggregation (chi tiết toán học)

SB3's SAC policy chứa các tensors:

```
policy.state_dict() keys:
  actor.latent_pi.0.weight  (256×obs_dim)   ← shared layers
  actor.latent_pi.0.bias    (256,)
  actor.latent_pi.2.weight  (256×256)
  actor.latent_pi.2.bias    (256,)
  actor.mu.weight           (act_dim×256)   ← mean head
  actor.mu.bias             (act_dim,)
  actor.log_std.weight      (act_dim×256)   ← std head  
  actor.log_std.bias        (act_dim,)
  critic.qf0.*              ...             ← Q1 network
  critic.qf1.*              ...             ← Q2 network
  critic_target.qf0.*       ...             ← target Q1
  critic_target.qf1.*       ...             ← target Q2
```

**FedAvg per tensor**:
$$w^{global}_k = \sum_{i=1}^{N} \alpha_i \cdot w^i_k, \quad \sum_{i=1}^N \alpha_i = 1$$

Với uniform aggregation: $\alpha_i = 1/N = 1/6 \approx 0.1\overline{6}$

**Ví dụ tính** với tensor `actor.latent_pi.0.bias` (256-dim):
```
Agent 1: b₁ = [0.12, -0.05, 0.33, ...]
Agent 2: b₂ = [0.08, -0.11, 0.28, ...]
...
Agent 6: b₆ = [0.15, -0.03, 0.31, ...]

b_global = (1/6) × (b₁ + b₂ + ... + b₆)
         = [0.11, -0.07, 0.30, ...]
```

In [9]:
class FedAvgAggregator:
    """
    Server-side FedAvg aggregation cho SAC policies.

    Tách biệt hoàn toàn với môi trường và agent cục bộ —
    chỉ xử lý model weights (state_dict tensors).
    """

    def __init__(self, agent_ids: list, weights: list = None):
        """
        agent_ids : danh sách tên agent (building names)
        weights   : trọng số đóng góp Σαᵢ=1, None = uniform
        """
        self.agent_ids = agent_ids
        N = len(agent_ids)
        if weights is None:
            self.weights = {aid: 1.0 / N for aid in agent_ids}
        else:
            assert len(weights) == N and abs(sum(weights) - 1.0) < 1e-6
            self.weights = dict(zip(agent_ids, weights))
        self.global_state_dict = None
        self.round_history     = []   # log per-round metrics

    # ------------------------------------------------------------------
    def aggregate(self, local_state_dicts: dict) -> dict:
        """
        Thực hiện FedAvg:
            w_global[k] = Σᵢ αᵢ · wᵢ[k]   ∀ tensor key k

        local_state_dicts : {agent_id: OrderedDict từ policy.state_dict()}
        returns           : aggregated state_dict
        """
        reference = list(local_state_dicts.values())[0]
        global_sd = {}

        for key in reference.keys():
            # Kiểm tra tensor type — chỉ aggregate float tensors
            # (int tensors như batch norm counts không aggregate)
            stacked = torch.stack([
                local_state_dicts[aid][key].float() * self.weights[aid]
                for aid in self.agent_ids
            ])
            global_sd[key] = stacked.sum(dim=0).to(reference[key].dtype)

        self.global_state_dict = global_sd
        return global_sd

    # ------------------------------------------------------------------
    def broadcast(self, agents: dict):
        """
        Ghi global weights vào tất cả agents.
        agents : {agent_id: SAC model}
        """
        assert self.global_state_dict is not None, \
            "Cần gọi aggregate() trước khi broadcast()."
        for aid, agent in agents.items():
            agent.policy.load_state_dict(self.global_state_dict)

    # ------------------------------------------------------------------
    def compute_weight_divergence(self, local_state_dicts: dict) -> float:
        """
        Đo độ phân kỳ giữa các agent:
            Δ = (1/N) Σᵢ ||wᵢ - w_global||₂ / ||w_global||₂

        Giá trị cao → agents học rất khác nhau (data heterogeneous).
        Giá trị thấp → agents đã đồng thuận tốt.
        """
        if self.global_state_dict is None:
            return float("nan")
        total_div = 0.0
        for aid in self.agent_ids:
            diff_norm = sum(
                (local_state_dicts[aid][k].float() -
                 self.global_state_dict[k].float()).norm().item()
                for k in self.global_state_dict
            )
            glob_norm = sum(
                self.global_state_dict[k].float().norm().item()
                for k in self.global_state_dict
            )
            total_div += diff_norm / (glob_norm + 1e-8)
        return total_div / len(self.agent_ids)

    # ------------------------------------------------------------------
    def log_round(self, round_idx: int, metrics: dict):
        """Lưu metrics của mỗi round để vẽ đồ thị hội tụ."""
        entry = {"round": round_idx, **metrics}
        self.round_history.append(entry)

    def get_history_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.round_history)

    # ------------------------------------------------------------------
    def save_global_weights(self, path: Path):
        """Lưu global weights (dùng cho resume hoặc deploy)."""
        torch.save(self.global_state_dict, path)
        print(f"  [Server] Global weights saved → {path.name}")

    def load_global_weights(self, path: Path):
        """Tải global weights từ file."""
        self.global_state_dict = torch.load(path, weights_only=True)
        print(f"  [Server] Global weights loaded ← {path.name}")


# Quick sanity check
print("FedAvgAggregator định nghĩa thành công.")
agg_test = FedAvgAggregator(["B1","B2","B3"], weights=[0.5, 0.3, 0.2])
print(f"  Test weights: {agg_test.weights}")

FedAvgAggregator định nghĩa thành công.
  Test weights: {'B1': 0.5, 'B2': 0.3, 'B3': 0.2}


In [10]:
# ================================================================
# SETUP: Environments & Agents
# ================================================================

def make_env(schema_dict, episode_time_steps):
    def _init():
        env = CityLearnEnv(
            schema=schema_dict, central_agent=True,
            episode_time_steps=episode_time_steps
        )
        env = NormalizedObservationWrapper(env)
        env = StableBaselines3Wrapper(env)
        env = Monitor(env)
        return env
    return _init

def get_single_building_schema(full_schema, building_name):
    schema = copy.deepcopy(full_schema)
    schema["buildings"] = {building_name: schema["buildings"][building_name]}
    # ⭐ Áp reward function mới (FIX: dùng `schema` không phải `schema_dict`)
    schema["reward_function"]["type"] = "citylearn.reward_function.SolarPenaltyAndComfortReward"
    return schema

# --- Tải schema ---
print("Đang tải schema...")
_tmp = CityLearnEnv(schema=BASE_SCHEMA)
FULL_SCHEMA    = copy.deepcopy(_tmp.schema)
BUILDING_NAMES = list(FULL_SCHEMA["buildings"].keys())[:NUM_HOUSES]
del _tmp

BUILDING_SCHEMAS = {
    n: get_single_building_schema(FULL_SCHEMA, n) for n in BUILDING_NAMES
}

# Chọn VecEnv class
VecEnvCls = SubprocVecEnv if USE_SUBPROC else DummyVecEnv

# --- Probe dims ---
_probe = make_env(BUILDING_SCHEMAS[BUILDING_NAMES[0]], EPISODE_LENGTH)()
OBS_DIM = _probe.observation_space.shape[0]
ACT_DIM = _probe.action_space.shape[0]
_probe.close()

print(f"Buildings : {BUILDING_NAMES}")
print(f"Obs dim   : {OBS_DIM}  |  Act dim : {ACT_DIM}")

def override_sac_hyperparams(agent):
    """Override hyperparams sau khi load (vì warm-start từ MARL có thể khác)."""
    agent.batch_size      = BATCH_SIZE
    agent.train_freq      = TRAIN_FREQ
    agent.gradient_steps  = GRADIENT_STEPS
    agent.learning_starts = LEARNING_STARTS
    # train_freq cần wrap thành TrainFreq object cho SB3
    from stable_baselines3.common.utils import get_schedule_fn
    from stable_baselines3.common.type_aliases import TrainFreq, TrainFrequencyUnit
    agent.train_freq = TrainFreq(TRAIN_FREQ, TrainFrequencyUnit.STEP)
    return agent

# --- Khởi tạo agents ---
agents   = {}
vec_envs = {}

n_warm = 0
n_resume = 0
n_new = 0

for name in BUILDING_NAMES:
    node_dir_local = LOCAL_MODEL_DIR / name
    node_dir_local.mkdir(parents=True, exist_ok=True)

    env_fns = [make_env(BUILDING_SCHEMAS[name], EPISODE_LENGTH)
               for _ in range(N_ENVS)]
    vec_env = VecEnvCls(env_fns)
    vec_envs[name] = vec_env

    # Ưu tiên Resume từ FedRL checkpoint local
    local_ckpts = [f for f in glob.glob(str(node_dir_local / "sac_*.zip"))
                   if "replay_buffer" not in f and "_final" not in f]
    marl_path = MARL_MODEL_DIR / name / f"sac_{name}_final.zip"

    if local_ckpts:
        latest = max(local_ckpts, key=os.path.getctime)
        agent = SAC.load(latest, env=vec_env,
                         tensorboard_log=str(LOCAL_TB_DIR),
                         device="cuda")
        agent = override_sac_hyperparams(agent)
        print(f"  [{name}] 🔄 Resume FedRL: {Path(latest).name}")
        n_resume += 1

    elif WARM_START_FROM_MARL and marl_path.exists():
        agent = SAC.load(str(marl_path), env=vec_env,
                         tensorboard_log=str(LOCAL_TB_DIR),
                         device="cuda")
        agent = override_sac_hyperparams(agent)
        print(f"  [{name}] ⭐ Warm-start MARL: {marl_path.name}")
        n_warm += 1

    else:
        agent = SAC(
            "MlpPolicy", vec_env,
            policy_kwargs=dict(net_arch=NET_ARCH),
            batch_size=BATCH_SIZE,
            buffer_size=BUFFER_SIZE,
            learning_starts=LEARNING_STARTS,
            train_freq=TRAIN_FREQ,
            gradient_steps=GRADIENT_STEPS,
            device="cuda",
            verbose=0,
            tensorboard_log=str(LOCAL_TB_DIR),
        )
        print(f"  [{name}] 🆕 Khởi tạo từ scratch")
        n_new += 1

    agents[name] = agent

aggregator = FedAvgAggregator(BUILDING_NAMES, weights=AGGREGATION_WEIGHTS)

print(f"\n{'='*50}")
print(f"  Tổng quan:")
print(f"    🔄 Resume FedRL  : {n_resume}")
print(f"    ⭐ Warm-start MARL: {n_warm}")
print(f"    🆕 From scratch  : {n_new}")
print(f"{'='*50}")
print(f"\nAggregation weights: {aggregator.weights}")
print(f"Device: {next(agents[BUILDING_NAMES[0]].policy.parameters()).device}")


Đang tải schema...


INFO:root:The dataset names DNE in cache. Will download from intelligent-environments-lab/CityLearn/tree/v2.5.0 GitHub repository and write to /root/.cache/citylearn/v2.5.0/dataset_names.json. Next time DataSet.get_dataset_names is called, it will read from cache unless DataSet.clear_cache is run first.
INFO:root:Go here /root/.cache/citylearn/v2.5.0/datasets/citylearn_challenge_2023_phase_3_1/schema.json 
INFO:root:The citylearn_challenge_2023_phase_3_1 dataset DNE in cache. Will download from intelligent-environments-lab/CityLearn/tree/v2.5.0 GitHub repository and write to /root/.cache/citylearn/v2.5.0/datasets. Next time DataSet.get_dataset('citylearn_challenge_2023_phase_3_1') is called, it will read from cache unless DataSet.clear_cache is run first.
INFO:root:The PV sizing data DNE in cache. Will download from intelligent-environments-lab/CityLearn/tree/v2.5.0 GitHub repository and write to /root/.cache/citylearn/v2.5.0/misc. Next time DataSet.get_pv_sizing_data is called, it wil

Buildings : ['Building_1', 'Building_2', 'Building_3', 'Building_4', 'Building_5', 'Building_6']
Obs dim   : 32  |  Act dim : 3


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  [Building_1] 🆕 Khởi tạo từ scratch


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  [Building_5] 🆕 Khởi tạo từ scratch


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  [Building_6] 🆕 Khởi tạo từ scratch

  Tổng quan:
    🔄 Resume FedRL  : 0
    ⭐ Warm-start MARL: 0
    🆕 From scratch  : 6

Aggregation weights: {'Building_1': 0.16666666666666666, 'Building_2': 0.16666666666666666, 'Building_3': 0.16666666666666666, 'Building_4': 0.16666666666666666, 'Building_5': 0.16666666666666666, 'Building_6': 0.16666666666666666}
Device: cuda:0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Federated Training Loop

### Thuật toán đầy đủ

```
FedRL-SAC (Algorithm):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input: R rounds, K episodes/round, N agents
Output: w_global (global policy)

1. Khởi tạo: w_global ← w_init  (hoặc MARL IL weights)

2. FOR round r = 1 to R:
   
   a. BROADCAST: wᵢ ← w_global  ∀i ∈ {1..N}
   
   b. LOCAL TRAIN (song song trong thực tế, tuần tự ở đây):
      FOR each agent i:
          wᵢ ← SAC.learn(K episodes, starting from wᵢ)
   
   c. COLLECT: {w₁, w₂, ..., wₙ}
   
   d. AGGREGATE:
          w_global ← Σᵢ αᵢ · wᵢ
   
   e. LOG: reward, divergence, KPI proxy
   
   f. SAVE checkpoint mỗi 5 rounds

3. RETURN w_global
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

### Lưu ý quan trọng về bước Broadcast

**Round 1**: Mỗi agent bắt đầu với MARL IL weights (warm-start).  
**Round 2+**: Mỗi agent **nhận lại** `w_global` từ server trước khi train.

Điều này khác biệt với MARL IL — ở đây agents "gặp nhau" mỗi $K$ episodes thông qua weight averaging, giải quyết non-stationarity.

In [11]:
# ================================================================
# FEDERATED TRAINING LOOP
# ================================================================
import time
import datetime
import subprocess
import json
import numpy as np

def rsync_to_drive():
    # Sync local cache -> Drive (mot chieu)
    try:
        subprocess.run(
            ["rsync", "-a", f"{LOCAL_MODEL_DIR}/", f"{MODEL_DIR}/"],
            check=True, timeout=600,
        )
        subprocess.run(
            ["rsync", "-a", f"{LOCAL_TB_DIR}/", f"{TENSORBOARD_DIR}/"],
            check=True, timeout=600,
        )
        return True
    except Exception as e:
        print(f"       ⚠ rsync failed: {e}")
        return False

# Paths: đọc/ghi ở LOCAL; Drive là mirror
global_weights_path = LOCAL_MODEL_DIR / "global_weights_latest.pt"
history_path        = LOCAL_MODEL_DIR / "round_history.json"
start_round         = 0

if global_weights_path.exists():
    aggregator.load_global_weights(global_weights_path)
    aggregator.broadcast(agents)
    if history_path.exists():
        with open(history_path) as f:
            aggregator.round_history = json.load(f)
        start_round = len(aggregator.round_history)
    print(f"Resume từ round {start_round}/{NUM_ROUNDS}")
else:
    local_sds = {n: agents[n].policy.state_dict() for n in BUILDING_NAMES}
    aggregator.aggregate(local_sds)
    aggregator.broadcast(agents)
    print("Round 0: FedAvg khởi tạo từ warm-start weights.")

print(f"\nBắt đầu training từ round {start_round + 1} → {NUM_ROUNDS}")
print(f"{'─'*62}")

# ⏱️ Biến lưu tổng thời gian chạy để tính trung bình
total_run_time = 0

for round_idx in range(start_round + 1, NUM_ROUNDS + 1):
    round_start_time = time.time()  # ⏱️ Bắt đầu bấm giờ cho vòng lặp hiện tại

    print(f"\n[Round {round_idx:>3}/{NUM_ROUNDS}]", end="  ")

    # (b) LOCAL TRAIN
    round_rewards = {}
    for name, agent in agents.items():
        agent.learn(
            total_timesteps=LOCAL_STEPS_PER_ROUND,
            reset_num_timesteps=False,
            # ⭐ FIX: XÓA log_interval=None — đó là tắt TB log, KHÔNG phải tắt console.
            # Output console đã được tắt bởi verbose=0 lúc init SAC.
            # Để default (= 4 episodes/log) cho TensorBoard ghi đầy đủ.
            tb_log_name=f"{SESSION_NAME}_{name}",
        )
        if hasattr(agent, "ep_info_buffer") and len(agent.ep_info_buffer) > 0:
            round_rewards[name] = np.mean([ep["r"] for ep in agent.ep_info_buffer])
        else:
            round_rewards[name] = float("nan")

    mean_reward = np.nanmean(list(round_rewards.values()))
    print(f"mean_ep_reward = {mean_reward:>8.2f}", end="  ")

    # (c)(d)(e)(f) Aggregate + divergence + broadcast
    local_sds = {n: agents[n].policy.state_dict() for n in BUILDING_NAMES}
    aggregator.aggregate(local_sds)
    divergence = aggregator.compute_weight_divergence(local_sds)
    print(f"divergence = {divergence:.4f}")
    aggregator.broadcast(agents)

    aggregator.log_round(round_idx, {
        "mean_reward": mean_reward,
        "weight_divergence": divergence,
        **{f"reward_{n}": round_rewards.get(n, float("nan"))
           for n in BUILDING_NAMES},
    })

    # CHECKPOINT LOCAL
    if round_idx % CHECKPOINT_EVERY == 0 or round_idx == NUM_ROUNDS:
        aggregator.save_global_weights(global_weights_path)
        for name, agent in agents.items():
            ckpt_path = LOCAL_MODEL_DIR / name / f"sac_{name}_round{round_idx:03d}.zip"
            ckpt_path.parent.mkdir(parents=True, exist_ok=True)
            agent.save(str(ckpt_path))
        with open(history_path, "w") as f:
            json.dump(aggregator.round_history, f, indent=2)
        print(f"       → Local checkpoint saved (round {round_idx})")

    # RSYNC to Drive
    if round_idx % RSYNC_EVERY == 0 or round_idx == NUM_ROUNDS:
        if rsync_to_drive():
            print(f"       → Synced to Drive ✓")

    # ⏱️ TÍNH TOÁN VÀ IN THỜI GIAN CÒN LẠI (ETA)
    round_time = time.time() - round_start_time
    total_run_time += round_time
    rounds_completed_this_run = round_idx - start_round

    avg_round_time = total_run_time / rounds_completed_this_run
    rounds_left = NUM_ROUNDS - round_idx
    eta_seconds = int(avg_round_time * rounds_left)
    eta_str = str(datetime.timedelta(seconds=eta_seconds))

    print(f"       ⏱️ TG vòng này: {round_time:.1f}s | Ước tính còn lại (ETA): {eta_str}")

print(f"\n{'='*62}")
print("  FEDERATED TRAINING COMPLETE")
print(f"{'='*62}")

# Final save per agent (local) + sync
for name, agent in agents.items():
    final_path = LOCAL_MODEL_DIR / name / f"sac_{name}_fedrl_final.zip"
    final_path.parent.mkdir(parents=True, exist_ok=True)
    agent.save(str(final_path))

print("  Syncing tất cả sang Drive...")
rsync_to_drive()
print("  Final models saved → Drive ✓")

Round 0: FedAvg khởi tạo từ warm-start weights.

Bắt đầu training từ round 1 → 30
──────────────────────────────────────────────────────────────

[Round   1/30]  mean_ep_reward = -23323.26  divergence = 0.4700
       ⏱️ TG vòng này: 611.1s | Ước tính còn lại (ETA): 4:55:23

[Round   2/30]  mean_ep_reward = -13193.28  divergence = 0.3086
       ⏱️ TG vòng này: 622.8s | Ước tính còn lại (ETA): 4:47:55

[Round   3/30]  mean_ep_reward = -12005.28  divergence = 0.1892
       ⏱️ TG vòng này: 628.2s | Ước tính còn lại (ETA): 4:39:19

[Round   4/30]  mean_ep_reward = -11749.08  divergence = 0.1290
       ⏱️ TG vòng này: 626.1s | Ước tính còn lại (ETA): 4:29:33

[Round   5/30]  mean_ep_reward = -11575.22  divergence = 0.1118
  [Server] Global weights saved → global_weights_latest.pt
       → Local checkpoint saved (round 5)
       ⏱️ TG vòng này: 627.9s | Ước tính còn lại (ETA): 4:19:40

[Round   6/30]  mean_ep_reward = -10866.96  divergence = 0.0874
       ⏱️ TG vòng này: 634.5s | Ước tính còn

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


       → Synced to Drive ✓
       ⏱️ TG vòng này: 632.0s | Ước tính còn lại (ETA): 3:29:14

[Round  11/30]  

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


mean_ep_reward = -5580.45  divergence = 0.0586
       ⏱️ TG vòng này: 631.0s | Ước tính còn lại (ETA): 3:18:52

[Round  12/30]  mean_ep_reward = -4822.31  divergence = 0.0560
       ⏱️ TG vòng này: 637.9s | Ước tính còn lại (ETA): 3:08:39

[Round  13/30]  mean_ep_reward = -4357.46  divergence = 0.0538
       ⏱️ TG vòng này: 634.9s | Ước tính còn lại (ETA): 2:58:18

[Round  14/30]  mean_ep_reward = -3864.69  divergence = 0.0523
       ⏱️ TG vòng này: 632.1s | Ước tính còn lại (ETA): 2:47:52

[Round  15/30]  mean_ep_reward = -3208.41  divergence = 0.0501
  [Server] Global weights saved → global_weights_latest.pt
       → Local checkpoint saved (round 15)
       ⏱️ TG vòng này: 632.1s | Ước tính còn lại (ETA): 2:37:25

[Round  16/30]  mean_ep_reward = -2732.96  divergence = 0.0484
       ⏱️ TG vòng này: 635.2s | Ước tính còn lại (ETA): 2:27:00

[Round  17/30]  mean_ep_reward = -2442.05  divergence = 0.0473
       ⏱️ TG vòng này: 635.3s | Ước tính còn lại (ETA): 2:16:34

[Round  18/30]  me

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


       → Synced to Drive ✓
       ⏱️ TG vòng này: 622.1s | Ước tính còn lại (ETA): 0:00:00

  FEDERATED TRAINING COMPLETE
  Syncing tất cả sang Drive...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  Final models saved → Drive ✓


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Convergence Analysis — Phân tích quá trình hội tụ

Theo dõi 2 chỉ số qua các round:

1. **Mean Episode Reward**: Tăng → policy đang cải thiện
2. **Weight Divergence** $\Delta$: Giảm → các agents đang đồng thuận

**Kỳ vọng lý thuyết**:
- Reward tăng dần và ổn định (không oscillate như MARL IL)
- Divergence giảm dần → bằng chứng FedAvg đang ép các agents về consensus

In [12]:
hist_df = aggregator.get_history_df()

if hist_df.empty:
    print("Chưa có dữ liệu training. Hãy chạy cell Training trước.")
else:
    print("=== Convergence Summary ===")
    print(hist_df[["round", "mean_reward", "weight_divergence"]].to_string(index=False))

    # ── Tìm round hội tụ (reward cải thiện < 1%) ──────────────
    rewards = hist_df["mean_reward"].values
    converged_round = None
    for i in range(5, len(rewards)):
        window = rewards[i-5:i]
        improvement = (window[-1] - window[0]) / (abs(window[0]) + 1e-8)
        if abs(improvement) < 0.01:
            converged_round = int(hist_df.iloc[i]["round"])
            break

    print(f"\nRound hội tụ ước tính : {converged_round if converged_round else 'Chưa hội tụ'}")
    print(f"Best mean reward      : {rewards.max():.2f} (round {hist_df.loc[rewards.argmax(), 'round']})")
    print(f"Final divergence      : {hist_df['weight_divergence'].iloc[-1]:.4f}")

=== Convergence Summary ===
 round   mean_reward  weight_divergence
     1 -23323.256616           0.469959
     2 -13193.283123           0.308571
     3 -12005.280920           0.189206
     4 -11749.077783           0.129021
     5 -11575.219092           0.111802
     6 -10866.956361           0.087409
     7  -9297.320764           0.076127
     8  -7798.008022           0.071603
     9  -6921.357520           0.066141
    10  -6237.871685           0.062058
    11  -5580.445801           0.058591
    12  -4822.313986           0.055963
    13  -4357.464683           0.053796
    14  -3864.690536           0.052255
    15  -3208.405748           0.050074
    16  -2732.962646           0.048358
    17  -2442.048114           0.047282
    18  -2211.403905           0.045662
    19  -2041.679109           0.044461
    20  -1853.110504           0.043134
    21  -1722.188861           0.041922
    22  -1628.141226           0.040227
    23  -1547.267586           0.039014
    24  -145

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import subprocess

print("=== Tìm thư mục FedRL ===")
result = subprocess.run(
    ["find", "/content", "/root", "-type", "d", "-name", "*fedrl*", "-o", "-name", "*frl*"],
    capture_output=True, text=True
)
print(result.stdout)

print("\n=== Tìm file model .zip ===")
result = subprocess.run(
    ["find", "/content", "/root", "-name", "sac_Building_*.zip"],
    capture_output=True, text=True
)
print(result.stdout)

print("\n=== Tìm file global weights ===")
result = subprocess.run(
    ["find", "/content", "/root", "-name", "global_weights*.pt", "-o", "-name", "*.pt"],
    capture_output=True, text=True
)
print(result.stdout)

print("\n=== Tìm TensorBoard logs ===")
result = subprocess.run(
    ["find", "/content", "/root", "-type", "d", "-name", "training_logs"],
    capture_output=True, text=True
)
print(result.stdout)

=== Tìm thư mục FedRL ===


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/content/models/fedrl_sac_solar_comfort_FROMSCRATCH
/content/_frl_cache
/content/_frl_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH
/content/_frl_nowarm_cache
/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH
/content/_frl_nowarm_cache/training_logs/fedrl_sac_solar_comfort_FROMSCRATCH_Building_3_0
/content/_frl_nowarm_cache/training_logs/fedrl_sac_solar_comfort_FROMSCRATCH_Building_1_0
/content/_frl_nowarm_cache/training_logs/fedrl_sac_solar_comfort_FROMSCRATCH_Building_2_0
/content/_frl_nowarm_cache/training_logs/fedrl_sac_solar_comfort_FROMSCRATCH_Building_6_0
/content/_frl_nowarm_cache/training_logs/fedrl_sac_solar_comfort_FROMSCRATCH_Building_4_0
/content/_frl_nowarm_cache/training_logs/fedrl_sac_solar_comfort_FROMSCRATCH_Building_5_0
/content/drive/MyDrive/BigData/notebooks/colab_setup_frl.ipynb
/content/drive/MyDrive/BigData/venv_frl
/content/drive/MyDrive/BigData/fedrl_final_20260424_185321
/content/drive/MyDrive/BigData/fedrl_final_20260424_185321/models/

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_1/sac_Building_1_round030.zip
/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_1/sac_Building_1_round005.zip
/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_1/sac_Building_1_round020.zip
/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_1/sac_Building_1_round010.zip
/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_1/sac_Building_1_round025.zip
/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_1/sac_Building_1_fedrl_final.zip
/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_1/sac_Building_1_round015.zip
/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_4/sac_Building_4_fedrl_final.zip
/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_4/sac_Building_4_round020.zip
/content/_frl

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/global_weights_latest.pt
/content/drive/MyDrive/BigData/fedrl_final_20260424_185321/models/fedrl_sac_fedavg/global_weights_latest.pt
/content/drive/MyDrive/BigData/fedrl_final_20260424_185500/models/fedrl_sac_fedavg/global_weights_latest.pt
/content/drive/MyDrive/BigData/xxxfedrl_solar_comfort/models/fedrl_sac_solar_comfort/global_weights_latest.pt
/content/drive/MyDrive/BigData/xxxfedrl_nowarm/models/fedrl_sac_solar_comfort_FROMSCRATCH/global_weights_latest.pt


=== Tìm TensorBoard logs ===
/content/_frl_cache/training_logs
/content/_frl_nowarm_cache/training_logs
/content/training_logs
/content/drive/MyDrive/BigData/training_logs
/content/drive/MyDrive/BigData/fedrl_final_20260424_185321/training_logs
/content/drive/MyDrive/BigData/fedrl_final_20260424_185500/training_logs
/content/drive/MyDrive/BigData/xxxmarl_solar_comfort/training_logs
/content/drive/MyDrive/BigData/fedrl_final_20260509_034732/training_logs
/con

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import subprocess
from pathlib import Path
from datetime import datetime

# Tạo thư mục đích trên Drive với timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = Path(f'/content/drive/MyDrive/BigData/fedrl_final_{timestamp}')
backup_dir.mkdir(parents=True, exist_ok=True)

print(f"📁 Thư mục đích: {backup_dir}\n")

# Copy models
print("=== Copy models ===")
result = subprocess.run(
    ["rsync", "-av", "--progress",
     "/content/_frl_cache/models/",
     f"{backup_dir}/models/"],
    capture_output=True, text=True
)
print(result.stdout[-1500:])

# Copy TensorBoard logs
print("\n=== Copy training_logs ===")
result = subprocess.run(
    ["rsync", "-av", "--progress",
     "/content/_frl_cache/training_logs/",
     f"{backup_dir}/training_logs/"],
    capture_output=True, text=True
)
print(result.stdout[-1500:])

print(f"\n✅ Backup hoàn tất tại: {backup_dir}")

In [ ]:
import subprocess

print("=== 6 Final models ===")
subprocess.run(["bash", "-c", f"ls -lh {backup_dir}/models/fedrl_sac_fedavg/Building_*/*_final.zip"])

print("\n=== Global weights ===")
subprocess.run(["ls", "-lh", f"{backup_dir}/models/fedrl_sac_fedavg/global_weights_latest.pt"])

print("\n=== TensorBoard logs ===")
subprocess.run(["bash", "-c", f"ls {backup_dir}/training_logs/"])

print("\n=== Tổng dung lượng ===")
subprocess.run(["du", "-sh", str(backup_dir)])

print("\n=== Số file .zip (phải là 36: 6 final + 30 checkpoints) ===")
subprocess.run(["bash", "-c", f"find {backup_dir} -name '*.zip' | wc -l"])

=== 6 Final models ===

=== Global weights ===

=== TensorBoard logs ===

=== Tổng dung lượng ===

=== Số file .zip (phải là 36: 6 final + 30 checkpoints) ===


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


CompletedProcess(args=['bash', '-c', "find /content/drive/MyDrive/BigData/fedrl_final_20260510_193337 -name '*.zip' | wc -l"], returncode=0)

In [ ]:
# ============================================================
# BACKUP ALL-IN-ONE CELL
# ============================================================
from google.colab import drive
import subprocess
import os
from pathlib import Path
from datetime import datetime

# 1. Mount Drive (nếu chưa mount)
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("✓ Drive đã mount sẵn")

# 2. Kiểm tra nguồn
SOURCE = "/content/_frl_cache"
if not os.path.exists(SOURCE):
    print(f"❌ KHÔNG tìm thấy nguồn: {SOURCE}")
    # Thử nguồn khác
    SOURCE = "/content"
    print(f"→ Thử dùng: {SOURCE}")

# 3. Tạo thư mục đích mới (timestamp mới)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP = f"/content/drive/MyDrive/BigData/fedrl_final_{timestamp}"
os.makedirs(BACKUP, exist_ok=True)
print(f"📁 Backup đến: {BACKUP}")

# 4. Copy models
print("\n=== Đang copy models... ===")
r1 = subprocess.run(
    ["rsync", "-av",
     f"{SOURCE}/models/",
     f"{BACKUP}/models/"],
    capture_output=True, text=True
)
if r1.returncode != 0:
    print(f"❌ Lỗi rsync models: {r1.stderr[-500:]}")
else:
    # Đếm số file copy
    n_files = r1.stdout.count('.zip')
    print(f"✓ Copy {n_files} file .zip")

# 5. Copy logs
print("\n=== Đang copy training_logs... ===")
r2 = subprocess.run(
    ["rsync", "-av",
     f"{SOURCE}/training_logs/",
     f"{BACKUP}/training_logs/"],
    capture_output=True, text=True
)
if r2.returncode != 0:
    print(f"❌ Lỗi rsync logs: {r2.stderr[-500:]}")
else:
    print("✓ Copy logs xong")

# 6. Verify (dùng path tuyệt đối, không dùng biến)
print(f"\n{'='*60}")
print("VERIFY KẾT QUẢ")
print(f"{'='*60}")

subprocess.run(["bash", "-c", f"""
echo "=== Cấu trúc thư mục ==="
ls -la {BACKUP}/

echo ""
echo "=== 6 Final models ==="
find {BACKUP} -name "*_fedrl_final.zip" -exec ls -lh {{}} \\;

echo ""
echo "=== Global weights ==="
find {BACKUP} -name "global_weights*.pt" -exec ls -lh {{}} \\;

echo ""
echo "=== Tổng dung lượng ==="
du -sh {BACKUP}

echo ""
echo "=== Đếm file .zip ==="
find {BACKUP} -name "*.zip" | wc -l
"""])

print(f"\n✅ HOÀN TẤT. Path backup: {BACKUP}")

✓ Drive đã mount sẵn
📁 Backup đến: /content/drive/MyDrive/BigData/fedrl_final_20260510_193337

=== Đang copy models... ===
✓ Copy 0 file .zip

=== Đang copy training_logs... ===
✓ Copy logs xong

VERIFY KẾT QUẢ

✅ HOÀN TẤT. Path backup: /content/drive/MyDrive/BigData/fedrl_final_20260510_193337


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Evaluation — Đánh giá FedRL trên joint environment

Quy trình đánh giá **giống hệt MARL IL**:
- Split joint obs 246-dim → 6 × 41-dim per agent
- Mỗi agent predict với `deterministic=True`
- Combine actions → 18-dim joint action

**Điểm khác biệt**: Agents FedRL đã được đồng bộ hóa qua `w_global` → kỳ vọng phối hợp tốt hơn MARL IL.

In [ ]:
# ================================================================
# EVALUATION
# ================================================================

joint_schema = copy.deepcopy(FULL_SCHEMA)
joint_schema["buildings"] = {n: joint_schema["buildings"][n]
                              for n in BUILDING_NAMES}

eval_env_raw  = CityLearnEnv(joint_schema, central_agent=True)
eval_env_norm = NormalizedObservationWrapper(eval_env_raw)
eval_env      = StableBaselines3Wrapper(eval_env_norm)

joint_obs_dim = eval_env.observation_space.shape[0]
joint_act_dim = eval_env.action_space.shape[0]
obs_per_agent = joint_obs_dim // NUM_HOUSES
act_per_agent = joint_act_dim // NUM_HOUSES

print(f"Joint obs : {joint_obs_dim}  |  obs/agent : {obs_per_agent}")
print(f"Joint act : {joint_act_dim}  |  act/agent : {act_per_agent}")

def resolve_final_ckpt(name):
    for base in (LOCAL_MODEL_DIR, MODEL_DIR):
        p = base / name / f"sac_{name}_fedrl_final.zip"
        if p.exists():
            return p
    return None

print("\nĐang tải FedRL final agents...")
eval_agents = {}
for name in BUILDING_NAMES:
    p = resolve_final_ckpt(name)
    if p is None:
        raise FileNotFoundError(
            f"Không tìm thấy sac_{name}_fedrl_final.zip ở cả "
            f"{LOCAL_MODEL_DIR} và {MODEL_DIR}. Hãy chạy Training trước."
        )
    eval_agents[name] = SAC.load(str(p))
    print(f"  ✓ {name}  ({p})")

obs, _ = eval_env.reset()
done = False
total_reward = 0.0
step_count   = 0

print("\nĐang chạy evaluation rollout...")
while not done:
    obs_per_building = [
        obs[i * obs_per_agent : (i+1) * obs_per_agent]
        for i in range(NUM_HOUSES)
    ]
    action_parts = []
    for i, name in enumerate(BUILDING_NAMES):
        a, _ = eval_agents[name].predict(
            obs_per_building[i].reshape(1, -1), deterministic=True
        )
        action_parts.append(a[0])

    joint_action = np.concatenate(action_parts).astype(np.float32)

    step_result = eval_env.step(joint_action)
    if len(step_result) == 5:
        obs, reward, terminated, truncated, _ = step_result
        done = terminated or truncated
    else:
        obs, reward, done, _ = step_result

    r = reward[0] if hasattr(reward, "__len__") else float(reward)
    total_reward += r
    step_count   += 1

print(f"Rollout hoàn tất — steps: {step_count:,}  |  total reward: {total_reward:.4f}")


Joint obs : 87  |  obs/agent : 14
Joint act : 18  |  act/agent : 3

Đang tải FedRL final agents...
  ✓ Building_1  (/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_1/sac_Building_1_fedrl_final.zip)
  ✓ Building_2  (/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_2/sac_Building_2_fedrl_final.zip)
  ✓ Building_3  (/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_3/sac_Building_3_fedrl_final.zip)
  ✓ Building_4  (/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_4/sac_Building_4_fedrl_final.zip)
  ✓ Building_5  (/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_5/sac_Building_5_fedrl_final.zip)
  ✓ Building_6  (/content/_frl_nowarm_cache/models/fedrl_sac_solar_comfort_FROMSCRATCH/Building_6/sac_Building_6_fedrl_final.zip)

Đang chạy evaluation rollout...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ValueError: Error: Unexpected observation shape (1, 14) for Box environment, please use (32,) or (n_env, 32) for the observation shape.

In [ ]:
# ================================================================
# KPI EXTRACTION
# ================================================================
kpis = eval_env_raw.evaluate()
kpis_pivot = (
    kpis.pivot(index="cost_function", columns="name", values="value")
    .round(4).dropna(how="all")
)
print("=== FedRL — KPIs ===")
display(kpis_pivot)

In [ ]:
# ================================================================
# BẢNG SO SÁNH TOÀN DIỆN (4 phương pháp)
# ================================================================
KEY_METRICS = [
    "cost_total",
    "discomfort_proportion",
    "carbon_emissions_total",
    "electricity_consumption_total",
    "zero_net_energy",
    "all_time_peak_average",
    "ramping_average",
]

BASELINES = {
    "RBC (1 nhà)": {
        "cost_total": 2.013, "discomfort_proportion": 0.984,
        "carbon_emissions_total": 2.141, "electricity_consumption_total": 2.121,
        "zero_net_energy": 2.199, "all_time_peak_average": 1.138,
        "ramping_average": 1.091,
    },
    "SAC Centralized\n(1 nhà)": {
        "cost_total": 0.8959, "discomfort_proportion": 0.0657,
        "carbon_emissions_total": 0.9293, "electricity_consumption_total": 0.9282,
        "zero_net_energy": 0.8210, "all_time_peak_average": 1.0313,
        "ramping_average": 1.9068,
    },
    "SAC Centralized\n(6 nhà)": {
        "cost_total": 0.7947, "discomfort_proportion": 0.5415,
        "carbon_emissions_total": 0.8016, "electricity_consumption_total": 0.7991,
        "zero_net_energy": 0.7427, "all_time_peak_average": 0.9460,
        "ramping_average": 1.1309,
    },
    "MARL IL\n(6 nhà)": {
        m: float("nan") for m in KEY_METRICS  # Điền sau khi chạy marl_sac.ipynb
    },
}

# Lấy kết quả FedRL từ kpis_pivot
fedrl_district = {}
for m in KEY_METRICS:
    try:
        fedrl_district[m] = kpis_pivot.loc[m, "District"]
    except KeyError:
        fedrl_district[m] = float("nan")
BASELINES["FedRL FedAvg\n(6 nhà) ← HERE"] = fedrl_district

comparison_df = pd.DataFrame(BASELINES, index=KEY_METRICS)
comparison_df.index.name = "KPI"

# Tính % cải thiện FedRL so với Centralized 6-nhà
central_6 = comparison_df["SAC Centralized\n(6 nhà)"]
fedrl_col  = comparison_df["FedRL FedAvg\n(6 nhà) ← HERE"]
improvement = ((central_6 - fedrl_col) / central_6.abs() * 100).round(2)
comparison_df["Δ vs Central-6 (%)"] = improvement.apply(
    lambda x: f"+{x:.1f}%" if x > 0 else f"{x:.1f}%"
)

def highlight_best(s):
    numeric = s.apply(lambda x: float(str(x).replace("%","").replace("+",""))
                      if isinstance(x, str) and any(c.isdigit() for c in str(x))
                      else (x if isinstance(x, (int, float)) else float("nan")))
    is_best = numeric == numeric.min()
    return ["background-color: #c8f7c5; font-weight: bold" if v else ""
            for v in is_best]

display(
    comparison_df.style
    .apply(highlight_best, axis=1,
           subset=[c for c in comparison_df.columns if "Δ" not in c])
    .format({c: "{:.4f}" for c in comparison_df.columns if "Δ" not in c},
            na_rep="(run marl_sac)")
    .set_caption("So sánh tất cả phương pháp — District KPIs (thấp hơn = tốt hơn)")
)

## Kết luận & Hướng tiếp theo

### FedRL vs các phương pháp khác

| Góc độ | RBC | Central SAC | MARL IL | **FedRL** |
|---|---|---|---|---|
| Privacy | ✓ Local | ✗ Share obs | ✓ Local | ✓ Only weights |
| Scalability | ✓ | ✗ Retrain all | ✓ | ✓ Add node |
| Coordination | ✗ | ✓ Joint policy | ✗ | ✓ Via FedAvg |
| Non-stationarity | N/A | N/A | ✗ | ✓ Synced |
| Communication | None | Continuous | None | Periodic |
| Data heterogeneity | N/A | N/A | ✓ | ~ (FedProx cần) |

### Hyperparameters ảnh hưởng lớn nhất

- **`NUM_ROUNDS`**: Nhiều round → hội tụ tốt hơn nhưng tốn thời gian
- **`LOCAL_EPISODES_PER_ROUND` (K)**: K nhỏ → sync thường xuyên (ít drift), K lớn → train sâu hơn mỗi round (trade-off)
- **`AGGREGATION_WEIGHTS`**: Có thể dùng proportional weights dựa trên lượng data hoặc performance của từng nhà

### Công thức trade-off K

$$\text{Communication cost} \propto \frac{1}{K} \qquad \text{Client drift} \propto K$$

Optimal $K^*$ phụ thuộc vào mức độ heterogeneity của data giữa các buildings.

### Bước tiếp theo (Giai đoạn 3+)

1. **FedProx**: Thêm proximal regularization để xử lý data heterogeneous
2. **Personalized FedRL**: Mỗi nhà giữ layer cuối riêng, share các layer đầu
3. **Differential Privacy**: Clip + Gaussian noise trước khi upload weights
4. **Async Federated RL**: Không cần chờ tất cả agents — phù hợp thực tế